# Independent PdM Reproduction Companion

This is a **runnable notebook** for the public v1.0.0 implementation. It is **not** a verified exact reproduction of the historical published Table 10. It intentionally has no prepopulated outputs: execute all cells with your independently downloaded AI4I data. The separate R15 author-audited notebook was executed successfully on Python 3.13; this compact GitHub companion is a new launcher and visualization interface, not a copy of R15's saved outputs.

## 1. Setup and dataset
Install dependencies from `requirements.txt`. Download AI4I 2020 from the original distributor and place `ai4i2020.csv` in `data/`. Start Jupyter from repository root or `notebooks/`.

In [ ]:
from pathlib import Path
import os, sys, platform, hashlib
import pandas as pd
ROOT = next((p for p in (Path.cwd().resolve(), Path.cwd().resolve().parent) if (p/'src'/'run_experiment.py').is_file()), None)
if ROOT is None: raise FileNotFoundError('Open from repository root or notebooks/')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
DATA=ROOT/'data'/'ai4i2020.csv'
if not DATA.exists(): raise FileNotFoundError('Download AI4I 2020 to data/ai4i2020.csv')
print('Python',platform.python_version(),'SHA256',hashlib.sha256(DATA.read_bytes()).hexdigest())
raw=pd.read_csv(DATA)
print('Shape',raw.shape,'Failure count',int(raw['Machine failure'].sum()))
display(raw.head())

## 2. Engineering-derived features
Use the canonical public implementation; no new features are invented.

In [ ]:
from src.features import engineer_features, ORIGINAL_FEATURES, FULL_FEATURES
engineered=engineer_features(raw)
display(engineered[['UDI']+[c for c in FULL_FEATURES if c not in ORIGINAL_FEATURES]].head())

## 3. Execute the benchmark
The original implementation controls the train/test split and model-specific imbalance handling. The held-out test set must not be used for parameter or threshold selection.

In [ ]:
import subprocess
run=subprocess.run([sys.executable,'-m','src.run_experiment'],cwd=ROOT,capture_output=True,text=True)
print(run.stdout[-3000:])
if run.returncode: raise RuntimeError(run.stderr[-4000:])

## 4. Inspect actual rerun metrics
These numbers are computed by your execution and should not be conflated with the published reference CSV.

In [ ]:
GEN=ROOT/'results'/'generated'
metrics=pd.read_csv(GEN/'benchmark_and_ablation_metrics.csv')
preds=pd.read_csv(GEN/'test_predictions.csv')
display(metrics)
print('Predictions:',len(preds))

## 5. Reproduce F1 ablation and a confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
ax=metrics.pivot(index='model',columns='feature_set',values='f1').plot(kind='bar',figsize=(9,4))
ax.set_ylabel('F1 score');plt.tight_layout();plt.show()
selected=preds[(preds['feature_set']=='Physics-informed')&(preds['model']=='XGBoost')]
ConfusionMatrixDisplay.from_predictions(selected['y_true'],selected['y_pred'])
plt.tight_layout();plt.show()

## 6. Published values versus independent rerun
The reference is transcribed from the article, **not** generated by the current execution.

In [ ]:
reference=pd.read_csv(ROOT/'results'/'published'/'table_10_benchmark_results.csv')
rerun=metrics[metrics['feature_set']=='Physics-informed'][['model','f1']].rename(columns={'f1':'rerun_f1'})
display(reference[['model','f1']].rename(columns={'f1':'published_f1'}).merge(rerun,on='model',how='left'))

## Interpretation and limitations
The public code uses one-hot machine type, rolling tool-wear statistics built before the random split, and model-specific imbalance procedures. Exact Table 10 agreement is not established. The AI4I data is synthetic and must be obtained independently. See `docs/REPRODUCIBILITY.md` and `docs/RESULTS_CROSSWALK.md`.